In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torchmetrics import Accuracy
from tqdm.notebook import tqdm

In [ ]:
train_data=pd.read_csv(r'Dataset/train.csv')
test_data=pd.read_csv(r'Dataset/test.csv')

In [ ]:
train_features=train_data.drop('label',axis=1)
test_features=test_data.drop('label',axis=1)

train_labels=train_data['label'].values
test_labels=test_data['label'].values

In [ ]:
def img_reshape_std(train,test):
    img_train=[]
    img_test=[]

    transposed_train=train.transpose()
    transposed_test=test.transpose()


    for idx in range(len(train)):
        val=transposed_train[idx].values
        val=val.reshape(28,28)
        img_train.append(val)


    for idx in range(len(test)):
        val=transposed_test[idx].values
        val=val.reshape(28,28)
        img_test.append(val)


    return img_train,img_test
        

In [ ]:
train_set_processed,test_set_processed=img_reshape_std(train_features,test_features)

In [ ]:
train_tensor=torch.tensor(train_set_processed,dtype=torch.float32)
test_tensor=torch.tensor(test_set_processed,dtype=torch.float32)

train_tensor_labels=torch.tensor(train_labels,dtype=torch.long)
test_tensor_labels=torch.tensor(test_labels,dtype=torch.long)

In [ ]:
train_dataset=TensorDataset(train_tensor,train_tensor_labels)
test_dataset=TensorDataset(test_tensor,test_tensor_labels)

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=256,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=256,shuffle=False)

In [ ]:
def layer(in_channels,out_channels,kernel_size,padding,batchnorm=True,stride=(1,1)):
    if batchnorm == True:
        sub_model=nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    stride=stride),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU()
        )
        return sub_model
    else:
        sub_model=nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    stride=stride),
                    nn.ReLU()
        )
        return sub_model


In [ ]:
model=nn.Sequential(
    layer(in_channels=1,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    layer(in_channels=16,out_channels=16,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),

    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),

    nn.Dropout(0.2),

    layer(in_channels=16,out_channels=32,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


      layer(in_channels=32,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),   


    nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)),


    layer(in_channels=64,out_channels=64,
          kernel_size=(3,3),
          padding=(1,1),batchnorm=True),


      nn.Dropout(0.2),

    nn.AdaptiveAvgPool2d(1),

    nn.Flatten(),

    nn.Linear(64,10)
)

In [ ]:
if torch.cuda.is_available():
    device='cuda'
else:
    device='cpu'
device

In [ ]:
model=model.to(device)

In [ ]:
loss=nn.CrossEntropyLoss()

In [ ]:
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [ ]:
c=0
for i in model.parameters():
    if i.requires_grad==True:
        c+=i.numel()
print(c)

In [ ]:
class Average:
    def __init__(self):
        self.val=0
        self.sum=0
        self.avg=0
        self.count=0
        
    def update(self,value):
        self.val=value
        self.sum +=value
        self.count+=1
        self.avg=self.sum/self.count

In [ ]:
hist_loss_train=[]
hist_acc_train=[]
hist_loss_test=[]
hist_acc_test=[]

In [ ]:
def train_one_epoch(model, train_data_loader, loss, optimizer, epoch):
    model.train()
    total_loss_train = Average()
    total_acc_train = Accuracy(task='multiclass',num_classes=10).to(device)

    with tqdm(train_data_loader) as t_data_loader:
        for x_batch , y_batch in t_data_loader:
            x_batch=x_batch.unsqueeze(1)
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_predict = model(x_batch)
            loss_train = loss(y_predict.squeeze(), y_batch)
            total_acc_train(y_predict.squeeze(), y_batch.int())

            loss_train.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss_train.update(loss_train.item())

            t_data_loader.set_postfix(loss=total_loss_train.avg, acc=total_acc_train.compute().item())
            t_data_loader.set_description(f"Train epoch = {epoch}")
            
        hist_loss_train.append(total_loss_train.avg)
        hist_acc_train.append(total_acc_train.compute().item())

In [ ]:
def evaluate(model,test_data_loader,loss,epoch):
    model.eval()
    total_loss_test = Average()
    total_acc_test = Accuracy(task='multiclass',num_classes=10).to(device)

    with torch.no_grad():
        with tqdm(test_data_loader, colour="red") as t_data_loader:
            for x_batch, y_batch in t_data_loader:
                x_batch=x_batch.unsqueeze(1)
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                y_predict = model(x_batch)
                loss_test = loss(y_predict.squeeze(), y_batch)
                total_acc_test(y_predict.squeeze(), y_batch.int())

                total_loss_test.update(loss_test.item())

                t_data_loader.set_postfix(loss=total_loss_test.avg, acc=total_acc_test.compute().item())
                t_data_loader.set_description(f"Test epoch = {epoch}")
                
            hist_loss_test.append(total_loss_test.avg)
            hist_acc_test.append(total_acc_test.compute().item())

In [ ]:
epochs=5
for epoch in range(epochs):
    train_one_epoch(model,train_loader,loss,optimizer,epoch+1)
    evaluate(model,test_loader,loss,epoch+1)

In [ ]:
# Train:
# Final Acc Train= 0.998 , Final Loss Train= 0.00673
# Final Acc Test= 0.999 , Final Loss Test= 0.00575
# Epochs= 5

In [ ]:
plt.plot(hist_loss_train,label='Train Loss')
plt.plot(hist_loss_test,label='Test Loss')
plt.grid(True)
plt.legend() 

In [ ]:
plt.plot(hist_acc_train,label='Train Acc')
plt.plot(hist_acc_test,label='Test Acc')
plt.grid(True)
plt.legend() 

In [ ]:
#torch.save(model.state_dict(),'Number_Predict_Finall.pth')